Import


In [1]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os
import sys
import base64
from IPython.display import display, Image, HTML
from sklearn.metrics import mean_absolute_error, r2_score
sys.path.append("../backend")
from vital_signs_analyzer import (
    preprocess_image, 
    extract_face_roi, 
    analyze_skin_color_variations, 
    estimate_bp,
    create_diagnostic_image
)

In [2]:
def test_analysis_on_image(image_path, known_hr=None):
    """Test the heart rate analysis on a sample image"""
    # Load the image
    img = cv2.imread(image_path)
    if img is None:
        print(f"Failed to load image: {image_path}")
        return
    
    # Display the image
    plt.figure(figsize=(10, 8))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title("Original Image")
    plt.axis('off')
    plt.show()
    
    # Initialize face cascade
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    
    # Extract face regions
    face_info = extract_face_roi(img, face_cascade)
    if face_info is None:
        print("No face detected in the image!")
        return
    
    # Display the face regions
    fig, axes = plt.subplots(1, 4, figsize=(15, 5))
    regions = ['face', 'forehead', 'left_cheek', 'right_cheek']
    titles = ['Face', 'Forehead', 'Left Cheek', 'Right Cheek']
    
    for i, region in enumerate(regions):
        axes[i].imshow(cv2.cvtColor(face_info[region], cv2.COLOR_BGR2RGB))
        axes[i].set_title(titles[i])
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Analyze the face
    analysis_result = analyze_skin_color_variations(face_info)
    
    # Create diagnostic image
    diagnostic_img = create_diagnostic_image(img, face_info)
    if diagnostic_img:
        # Convert base64 to image for display
        decoded = base64.b64decode(diagnostic_img)
        plt.figure(figsize=(10, 8))
        plt.imshow(plt.imread(io.BytesIO(decoded)))
        plt.title("Diagnostic Image")
        plt.axis('off')
        plt.show()
    
    # Display the analysis results
    print(f"Estimated Heart Rate: {analysis_result['heart_rate']} BPM")
    print(f"Heart Rate Range: {analysis_result['heart_rate_range'][0]}-{analysis_result['heart_rate_range'][1]} BPM")
    print(f"Confidence: {analysis_result['confidence']}%")
    
    # Calculate error if known heart rate is provided
    if known_hr:
        error = abs(analysis_result['heart_rate'] - known_hr)
        error_pct = error / known_hr * 100
        print(f"\nKnown Heart Rate: {known_hr} BPM")
        print(f"Absolute Error: {error} BPM")
        print(f"Relative Error: {error_pct:.2f}%")
        print(f"Within Error Range: {'Yes' if known_hr >= analysis_result['heart_rate_range'][0] and known_hr <= analysis_result['heart_rate_range'][1] else 'No'}")
    
    # Display the detailed metrics
    if 'metrics' in analysis_result:
        print("\nDetailed Metrics:")
        for key, value in analysis_result['metrics'].items():
            print(f"  {key}: {value}")
    
    # Estimate blood pressure with demographics
    age = 30  # Example age
    weight_kg = 70  # Example weight
    height_cm = 170  # Example height
    is_male = True  # Example gender
    
    sys_bp, dia_bp = estimate_bp(analysis_result['heart_rate'], age, weight_kg, height_cm, is_male)
    print(f"\nEstimated Blood Pressure: {sys_bp}/{dia_bp} mmHg")
    
    # Visualize the results
    plt.figure(figsize=(12, 6))
    
    # Heart rate visualization
    plt.subplot(1, 2, 1)
    plt.bar(['Heart Rate'], [analysis_result['heart_rate']], color='blue', alpha=0.7)
    plt.errorbar(['Heart Rate'], [analysis_result['heart_rate']], 
                yerr=[[analysis_result['heart_rate'] - analysis_result['heart_rate_range'][0]], 
                      [analysis_result['heart_rate_range'][1] - analysis_result['heart_rate']]], 
                fmt='o', color='red', ecolor='red', capsize=10)
    if known_hr:
        plt.axhline(y=known_hr, color='green', linestyle='--', label=f'Known HR: {known_hr}')
        plt.legend()
    plt.title('Heart Rate Estimation')
    plt.ylabel('BPM')
    plt.ylim(max(40, min(analysis_result['heart_rate_range'][0]-10, known_hr-10 if known_hr else 0)), 
             min(140, max(analysis_result['heart_rate_range'][1]+10, known_hr+10 if known_hr else 150)))
    
    # Blood pressure visualization
    plt.subplot(1, 2, 2)
    plt.bar(['Systolic', 'Diastolic'], [sys_bp, dia_bp], color=['orange', 'green'], alpha=0.7)
    plt.title('Blood Pressure Estimation')
    plt.ylabel('mmHg')
    
    plt.tight_layout()
    plt.show()
    
    return analysis_result

# Test with a sample image
# test_analysis_on_image('path/to/your/test/image.jpg', known_hr=75)

# Third cell - Parameter tuning
def parameter_tuning():
    """Test different parameter combinations to find optimal settings"""
    # Load a set of test images and known heart rates
    test_cases = [
        {'image': 'path/to/test1.jpg', 'hr': 65},
        {'image': 'path/to/test2.jpg', 'hr': 72},
        {'image': 'path/to/test3.jpg', 'hr': 85}
    ]
    
    # Parameters to vary
    rg_ratios = [20, 25, 30, 35]
    std_weights = [0.3, 0.5, 0.7, 1.0]
    
    results = []
    
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    
    for test_case in test_cases:
        img = cv2.imread(test_case['image'])
        face_info = extract_face_roi(img, face_cascade)
        
        if not face_info:
            print(f"No face detected in {test_case['image']}")
            continue
            
        for rg_ratio in rg_ratios:
            for std_weight in std_weights:
                # Modify the parameters for testing
                # This would require refactoring the analyze_skin_color_variations function
                # to accept parameters, or modifying a copy of it here
                
                # For now, just call the function and track results
                analysis_result = analyze_skin_color_variations(face_info)
                
                error = abs(analysis_result['heart_rate'] - test_case['hr'])
                
                results.append({
                    'image': test_case['image'],
                    'known_hr': test_case['hr'],
                    'estimated_hr': analysis_result['heart_rate'],
                    'rg_ratio': rg_ratio,
                    'std_weight': std_weight,
                    'error': error,
                    'confidence': analysis_result['confidence']
                })
    
    # Convert to DataFrame and find the best parameters
    df = pd.DataFrame(results)
    best_params = df.sort_values('error').groupby(['rg_ratio', 'std_weight']).mean().reset_index().sort_values('error')
    
    print("Best parameter combinations:")
    display(best_params.head(5))
    
    # Visualize error by parameter combination
    plt.figure(figsize=(12, 8))
    pivot = df.pivot_table(values='error', index='rg_ratio', columns='std_weight', aggfunc='mean')
    sns.heatmap(pivot, annot=True, cmap='YlGnBu_r', fmt='.1f')
    plt.title('Average Error by Parameter Combination')
    plt.tight_layout()
    plt.show()
    
    return best_params

# parameter_tuning()

# Fourth cell - Batch testing
def batch_test(test_dir, results_file='test_results.csv'):
    """Run the analysis on all images in a directory"""
    # Find all image files
    image_files = []
    for ext in ['jpg', 'jpeg', 'png']:
        image_files.extend(glob.glob(os.path.join(test_dir, f'*.{ext}')))
    
    if not image_files:
        print(f"No image files found in {test_dir}")
        return
    
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    
    results = []
    
    for img_file in image_files:
        print(f"Processing {img_file}...")
        img = cv2.imread(img_file)
        
        if img is None:
            print(f"  Failed to load image")
            continue
            
        face_info = extract_face_roi(img, face_cascade)
        
        if face_info is None:
            print(f"  No face detected")
            continue
            
        analysis_result = analyze_skin_color_variations(face_info)
        
        sys_bp, dia_bp = estimate_bp(analysis_result['heart_rate'])
        
        results.append({
            'file': os.path.basename(img_file),
            'heart_rate': analysis_result['heart_rate'],
            'confidence': analysis_result['confidence'],
            'systolic_bp': sys_bp,
            'diastolic_bp': dia_bp
        })
        
        print(f"  Heart Rate: {analysis_result['heart_rate']} BPM")
        print(f"  Confidence: {analysis_result['confidence']}%")
        print(f"  Blood Pressure: {sys_bp}/{dia_bp} mmHg")
    
    # Save results to CSV
    df = pd.DataFrame(results)
    df.to_csv(os.path.join(test_dir, results_file), index=False)
    
    print(f"\nProcessed {len(results)} images")
    print(f"Results saved to {os.path.join(test_dir, results_file)}")
    
    # Display statistics
    print("\nHeart Rate Statistics:")
    print(f"  Mean: {df['heart_rate'].mean():.1f} BPM")
    print(f"  Median: {df['heart_rate'].median():.1f} BPM")
    print(f"  Min: {df['heart_rate'].min()} BPM")
    print(f"  Max: {df['heart_rate'].max()} BPM")
    
    # Plot distributions
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    sns.histplot(df['heart_rate'], kde=True)
    plt.title('Heart Rate Distribution')
    plt.xlabel('BPM')
    
    plt.subplot(1, 3, 2)
    sns.histplot(df['confidence'], kde=True)
    plt.title('Confidence Distribution')
    plt.xlabel('Confidence (%)')
    
    plt.subplot(1, 3, 3)
    sns.scatterplot(x='heart_rate', y='confidence', data=df)
    plt.title('Heart Rate vs Confidence')
    plt.xlabel('Heart Rate (BPM)')
    plt.ylabel('Confidence (%)')
    
    plt.tight_layout()
    plt.show()
    
    return df


In [2]:
# Add this cell after your existing imports
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import numpy as np
import io
import glob
from scipy import stats

def evaluate_system_accuracy(ground_truth_data, test_dir=None):
    """
    Evaluate the accuracy and precision of the vital signs monitoring system
    
    Parameters:
    -----------
    ground_truth_data : dict or DataFrame
        Dictionary or DataFrame containing file names and known values for heart rate and BP
        Format: {filename: {'hr': value, 'systolic': value, 'diastolic': value}}
        Or DataFrame with columns ['file', 'known_hr', 'known_systolic', 'known_diastolic']
    
    test_dir : str
        Directory containing the test images
    
    Returns:
    --------
    dict
        Dictionary containing all evaluation metrics
    """
    if isinstance(ground_truth_data, dict):
        # Convert dictionary to DataFrame for easier processing
        gt_data = []
        for filename, values in ground_truth_data.items():
            gt_data.append({
                'file': filename,
                'known_hr': values.get('hr'),
                'known_systolic': values.get('systolic'),
                'known_diastolic': values.get('diastolic')
            })
        ground_truth_df = pd.DataFrame(gt_data)
    else:
        ground_truth_df = ground_truth_data.copy()
    
    # Process each image and get results
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    results = []
    
    # Get the list of image files if a directory is provided
    if test_dir:
        image_files = []
        for ext in ['jpg', 'jpeg', 'png']:
            image_files.extend(glob.glob(os.path.join(test_dir, f'*.{ext}')))
    else:
        # Use the filenames from ground truth data
        image_files = [os.path.join(test_dir if test_dir else '', f) for f in ground_truth_df['file']]
    
    for img_file in image_files:
        filename = os.path.basename(img_file)
        print(f"Processing {filename}...")
        
        # Find ground truth for this file
        gt_row = ground_truth_df[ground_truth_df['file'] == filename]
        if gt_row.empty:
            print(f"  No ground truth data for {filename}, skipping")
            continue
        
        known_hr = gt_row['known_hr'].values[0] if 'known_hr' in gt_row.columns else None
        known_systolic = gt_row['known_systolic'].values[0] if 'known_systolic' in gt_row.columns else None
        known_diastolic = gt_row['known_diastolic'].values[0] if 'known_diastolic' in gt_row.columns else None
        
        # Process the image
        img = cv2.imread(img_file)
        if img is None:
            print(f"  Failed to load image")
            continue
        
        face_info = extract_face_roi(img, face_cascade)
        if face_info is None:
            print(f"  No face detected")
            continue
        
        # Analyze the image
        analysis_result = analyze_skin_color_variations(face_info)
        
        # Get demographics if available
        age = gt_row['age'].values[0] if 'age' in gt_row.columns else 30
        weight_kg = gt_row['weight_kg'].values[0] if 'weight_kg' in gt_row.columns else 70
        height_cm = gt_row['height_cm'].values[0] if 'height_cm' in gt_row.columns else 170
        is_male = gt_row['is_male'].values[0] if 'is_male' in gt_row.columns else True
        
        # Estimate BP
        sys_bp, dia_bp = estimate_bp(analysis_result['heart_rate'], age, weight_kg, height_cm, is_male)
        
        # Calculate errors
        hr_error = abs(analysis_result['heart_rate'] - known_hr) if known_hr is not None else None
        sys_error = abs(sys_bp - known_systolic) if known_systolic is not None else None
        dia_error = abs(dia_bp - known_diastolic) if known_diastolic is not None else None
        
        # Determine if the estimation is within acceptable clinical margins
        # Typically ±5 BPM for HR, ±10 mmHg for systolic, ±5 mmHg for diastolic
        hr_within_margin = hr_error is not None and hr_error <= 5
        sys_within_margin = sys_error is not None and sys_error <= 10
        dia_within_margin = dia_error is not None and dia_error <= 5
        
        # Store result
        results.append({
            'file': filename,
            'estimated_hr': analysis_result['heart_rate'],
            'known_hr': known_hr,
            'hr_error': hr_error,
            'hr_within_margin': hr_within_margin,
            
            'estimated_systolic': sys_bp,
            'known_systolic': known_systolic,
            'systolic_error': sys_error,
            'systolic_within_margin': sys_within_margin,
            
            'estimated_diastolic': dia_bp,
            'known_diastolic': known_diastolic,
            'diastolic_error': dia_error,
            'diastolic_within_margin': dia_within_margin,
            
            'confidence': analysis_result['confidence']
        })
        
        print(f"  Heart Rate: Estimated={analysis_result['heart_rate']} BPM, Known={known_hr} BPM, Error={hr_error} BPM")
        if known_systolic and known_diastolic:
            print(f"  Blood Pressure: Estimated={sys_bp}/{dia_bp} mmHg, Known={known_systolic}/{known_diastolic} mmHg")
    
    # Convert results to DataFrame
    results_df = pd.DataFrame(results)
    
    # Calculate overall metrics
    metrics = {}
    
    # Heart Rate Metrics
    hr_results = results_df.dropna(subset=['hr_error'])
    if not hr_results.empty:
        metrics['heart_rate'] = {
            'MAE': mean_absolute_error(hr_results['known_hr'], hr_results['estimated_hr']),
            'RMSE': np.sqrt(mean_squared_error(hr_results['known_hr'], hr_results['estimated_hr'])),
            'R2': r2_score(hr_results['known_hr'], hr_results['estimated_hr']),
            'Mean Error': hr_results['hr_error'].mean(),
            'Median Error': hr_results['hr_error'].median(),
            'Max Error': hr_results['hr_error'].max(),
            'Within Margin': hr_results['hr_within_margin'].mean() * 100,
            'Pearson Correlation': stats.pearsonr(hr_results['known_hr'], hr_results['estimated_hr'])[0]
        }
    
    # Systolic BP Metrics
    sys_results = results_df.dropna(subset=['systolic_error'])
    if not sys_results.empty:
        metrics['systolic_bp'] = {
            'MAE': mean_absolute_error(sys_results['known_systolic'], sys_results['estimated_systolic']),
            'RMSE': np.sqrt(mean_squared_error(sys_results['known_systolic'], sys_results['estimated_systolic'])),
            'R2': r2_score(sys_results['known_systolic'], sys_results['estimated_systolic']),
            'Mean Error': sys_results['systolic_error'].mean(),
            'Median Error': sys_results['systolic_error'].median(),
            'Max Error': sys_results['systolic_error'].max(),
            'Within Margin': sys_results['systolic_within_margin'].mean() * 100,
            'Pearson Correlation': stats.pearsonr(sys_results['known_systolic'], sys_results['estimated_systolic'])[0]
        }
    
    # Diastolic BP Metrics
    dia_results = results_df.dropna(subset=['diastolic_error'])
    if not dia_results.empty:
        metrics['diastolic_bp'] = {
            'MAE': mean_absolute_error(dia_results['known_diastolic'], dia_results['estimated_diastolic']),
            'RMSE': np.sqrt(mean_squared_error(dia_results['known_diastolic'], dia_results['estimated_diastolic'])),
            'R2': r2_score(dia_results['known_diastolic'], dia_results['estimated_diastolic']),
            'Mean Error': dia_results['diastolic_error'].mean(),
            'Median Error': dia_results['diastolic_error'].median(),
            'Max Error': dia_results['diastolic_error'].max(),
            'Within Margin': dia_results['diastolic_within_margin'].mean() * 100,
            'Pearson Correlation': stats.pearsonr(dia_results['known_diastolic'], dia_results['estimated_diastolic'])[0]
        }
    
    # Display results
    print("\n=== SYSTEM EVALUATION RESULTS ===\n")
    
    if 'heart_rate' in metrics:
        print("HEART RATE METRICS:")
        print(f"  Mean Absolute Error (MAE): {metrics['heart_rate']['MAE']:.2f} BPM")
        print(f"  Root Mean Square Error (RMSE): {metrics['heart_rate']['RMSE']:.2f} BPM")
        print(f"  R² Score: {metrics['heart_rate']['R2']:.4f}")
        print(f"  Pearson Correlation: {metrics['heart_rate']['Pearson Correlation']:.4f}")
        print(f"  Mean Error: {metrics['heart_rate']['Mean Error']:.2f} BPM")
        print(f"  Median Error: {metrics['heart_rate']['Median Error']:.2f} BPM")
        print(f"  Maximum Error: {metrics['heart_rate']['Max Error']:.2f} BPM")
        print(f"  Measurements Within Acceptable Margin (±5 BPM): {metrics['heart_rate']['Within Margin']:.1f}%")
    
    if 'systolic_bp' in metrics:
        print("\nSYSTOLIC BLOOD PRESSURE METRICS:")
        print(f"  Mean Absolute Error (MAE): {metrics['systolic_bp']['MAE']:.2f} mmHg")
        print(f"  Root Mean Square Error (RMSE): {metrics['systolic_bp']['RMSE']:.2f} mmHg")
        print(f"  R² Score: {metrics['systolic_bp']['R2']:.4f}")
        print(f"  Pearson Correlation: {metrics['systolic_bp']['Pearson Correlation']:.4f}")
        print(f"  Mean Error: {metrics['systolic_bp']['Mean Error']:.2f} mmHg")
        print(f"  Median Error: {metrics['systolic_bp']['Median Error']:.2f} mmHg")
        print(f"  Maximum Error: {metrics['systolic_bp']['Max Error']:.2f} mmHg")
        print(f"  Measurements Within Acceptable Margin (±10 mmHg): {metrics['systolic_bp']['Within Margin']:.1f}%")
    
    if 'diastolic_bp' in metrics:
        print("\nDIASTOLIC BLOOD PRESSURE METRICS:")
        print(f"  Mean Absolute Error (MAE): {metrics['diastolic_bp']['MAE']:.2f} mmHg")
        print(f"  Root Mean Square Error (RMSE): {metrics['diastolic_bp']['RMSE']:.2f} mmHg")
        print(f"  R² Score: {metrics['diastolic_bp']['R2']:.4f}")
        print(f"  Pearson Correlation: {metrics['diastolic_bp']['Pearson Correlation']:.4f}")
        print(f"  Mean Error: {metrics['diastolic_bp']['Mean Error']:.2f} mmHg")
        print(f"  Median Error: {metrics['diastolic_bp']['Median Error']:.2f} mmHg")
        print(f"  Maximum Error: {metrics['diastolic_bp']['Max Error']:.2f} mmHg")
        print(f"  Measurements Within Acceptable Margin (±5 mmHg): {metrics['diastolic_bp']['Within Margin']:.1f}%")
    
    # Create visualizations
    if not hr_results.empty:
        plt.figure(figsize=(15, 12))
        
        # Scatter plot with regression line for heart rate
        plt.subplot(2, 2, 1)
        sns.regplot(x='known_hr', y='estimated_hr', data=hr_results, scatter_kws={'alpha':0.5})
        plt.plot([hr_results['known_hr'].min(), hr_results['known_hr'].max()], 
                 [hr_results['known_hr'].min(), hr_results['known_hr'].max()], 
                 'r--', label='Perfect Prediction')
        plt.title('Estimated vs. Known Heart Rate')
        plt.xlabel('Known Heart Rate (BPM)')
        plt.ylabel('Estimated Heart Rate (BPM)')
        plt.legend()
        
        # Bland-Altman plot for heart rate
        plt.subplot(2, 2, 2)
        mean_hr = (hr_results['estimated_hr'] + hr_results['known_hr']) / 2
        diff_hr = hr_results['estimated_hr'] - hr_results['known_hr']
        
        plt.scatter(mean_hr, diff_hr, alpha=0.5)
        plt.axhline(diff_hr.mean(), color='red', linestyle='-', label=f'Mean bias: {diff_hr.mean():.2f}')
        plt.axhline(diff_hr.mean() + 1.96 * diff_hr.std(), color='gray', linestyle='--', 
                   label=f'Upper 95% limit: {(diff_hr.mean() + 1.96 * diff_hr.std()):.2f}')
        plt.axhline(diff_hr.mean() - 1.96 * diff_hr.std(), color='gray', linestyle='--',
                   label=f'Lower 95% limit: {(diff_hr.mean() - 1.96 * diff_hr.std()):.2f}')
        plt.title('Bland-Altman Plot - Heart Rate')
        plt.xlabel('Mean of Known and Estimated HR (BPM)')
        plt.ylabel('Difference (Estimated - Known) (BPM)')
        plt.legend()
        
        # Error distribution histogram
        plt.subplot(2, 2, 3)
        sns.histplot(hr_results['hr_error'], kde=True)
        plt.axvline(hr_results['hr_error'].mean(), color='red', linestyle='-', label=f'Mean: {hr_results["hr_error"].mean():.2f}')
        plt.title('Heart Rate Error Distribution')
        plt.xlabel('Absolute Error (BPM)')
        plt.ylabel('Frequency')
        plt.legend()
        
        # Error vs. Confidence
        plt.subplot(2, 2, 4)
        sns.scatterplot(x='confidence', y='hr_error', data=hr_results)
        plt.title('Error vs. Confidence')
        plt.xlabel('Confidence (%)')
        plt.ylabel('Absolute Error (BPM)')
        
        plt.tight_layout()
        plt.show()
    
    # Similar visualizations for BP if available
    if not sys_results.empty and not dia_results.empty:
        plt.figure(figsize=(15, 12))
        
        # Scatter plot for systolic BP
        plt.subplot(2, 3, 1)
        sns.regplot(x='known_systolic', y='estimated_systolic', data=sys_results, scatter_kws={'alpha':0.5})
        plt.plot([sys_results['known_systolic'].min(), sys_results['known_systolic'].max()], 
                 [sys_results['known_systolic'].min(), sys_results['known_systolic'].max()], 
                 'r--', label='Perfect Prediction')
        plt.title('Estimated vs. Known Systolic BP')
        plt.xlabel('Known Systolic BP (mmHg)')
        plt.ylabel('Estimated Systolic BP (mmHg)')
        plt.legend()
        
        # Scatter plot for diastolic BP
        plt.subplot(2, 3, 2)
        sns.regplot(x='known_diastolic', y='estimated_diastolic', data=dia_results, scatter_kws={'alpha':0.5})
        plt.plot([dia_results['known_diastolic'].min(), dia_results['known_diastolic'].max()], 
                 [dia_results['known_diastolic'].min(), dia_results['known_diastolic'].max()], 
                 'r--', label='Perfect Prediction')
        plt.title('Estimated vs. Known Diastolic BP')
        plt.xlabel('Known Diastolic BP (mmHg)')
        plt.ylabel('Estimated Diastolic BP (mmHg)')
        plt.legend()
        
        # Combined BP scatter plot
        plt.subplot(2, 3, 3)
        plt.scatter(sys_results['known_systolic'], sys_results['estimated_systolic'], 
                   alpha=0.5, label='Systolic', color='red')
        plt.scatter(dia_results['known_diastolic'], dia_results['estimated_diastolic'], 
                   alpha=0.5, label='Diastolic', color='blue')
        plt.plot([60, 180], [60, 180], 'k--', label='Perfect Prediction')
        plt.title('Estimated vs. Known Blood Pressure')
        plt.xlabel('Known BP (mmHg)')
        plt.ylabel('Estimated BP (mmHg)')
        plt.legend()
        
        # Bland-Altman plot for systolic BP
        plt.subplot(2, 3, 4)
        mean_sys = (sys_results['estimated_systolic'] + sys_results['known_systolic']) / 2
        diff_sys = sys_results['estimated_systolic'] - sys_results['known_systolic']
        
        plt.scatter(mean_sys, diff_sys, alpha=0.5, color='red')
        plt.axhline(diff_sys.mean(), color='red', linestyle='-', label=f'Mean bias: {diff_sys.mean():.2f}')
        plt.axhline(diff_sys.mean() + 1.96 * diff_sys.std(), color='gray', linestyle='--', 
                   label=f'Upper 95% limit: {(diff_sys.mean() + 1.96 * diff_sys.std()):.2f}')
        plt.axhline(diff_sys.mean() - 1.96 * diff_sys.std(), color='gray', linestyle='--',
                   label=f'Lower 95% limit: {(diff_sys.mean() - 1.96 * diff_sys.std()):.2f}')
        plt.title('Bland-Altman Plot - Systolic BP')
        plt.xlabel('Mean of Known and Estimated (mmHg)')
        plt.ylabel('Difference (Estimated - Known) (mmHg)')
        plt.legend()
        
        # Bland-Altman plot for diastolic BP
        plt.subplot(2, 3, 5)
        mean_dia = (dia_results['estimated_diastolic'] + dia_results['known_diastolic']) / 2
        diff_dia = dia_results['estimated_diastolic'] - dia_results['known_diastolic']
        
        plt.scatter(mean_dia, diff_dia, alpha=0.5, color='blue')
        plt.axhline(diff_dia.mean(), color='blue', linestyle='-', label=f'Mean bias: {diff_dia.mean():.2f}')
        plt.axhline(diff_dia.mean() + 1.96 * diff_dia.std(), color='gray', linestyle='--', 
                   label=f'Upper 95% limit: {(diff_dia.mean() + 1.96 * diff_dia.std()):.2f}')
        plt.axhline(diff_dia.mean() - 1.96 * diff_dia.std(), color='gray', linestyle='--',
                   label=f'Lower 95% limit: {(diff_dia.mean() - 1.96 * diff_dia.std()):.2f}')
        plt.title('Bland-Altman Plot - Diastolic BP')
        plt.xlabel('Mean of Known and Estimated (mmHg)')
        plt.ylabel('Difference (Estimated - Known) (mmHg)')
        plt.legend()
        
        # BP Error distribution
        plt.subplot(2, 3, 6)
        sns.histplot(sys_results['systolic_error'], kde=True, color='red', label='Systolic', alpha=0.5)
        sns.histplot(dia_results['diastolic_error'], kde=True, color='blue', label='Diastolic', alpha=0.5)
        plt.title('Blood Pressure Error Distribution')
        plt.xlabel('Absolute Error (mmHg)')
        plt.ylabel('Frequency')
        plt.legend()
        
        plt.tight_layout()
        plt.show()
    
    return {
        'metrics': metrics,
        'results_df': results_df
    }

# Function to create sample ground truth data from measurements
def create_ground_truth_data():
    """Create a sample dataset with ground truth measurements"""
    return pd.DataFrame({
        'file': [f'sample{i}.jpg' for i in range(1, 21)],
        'known_hr': np.random.randint(60, 100, 20),
        'known_systolic': np.random.randint(110, 150, 20),
        'known_diastolic': np.random.randint(60, 90, 20),
        'age': np.random.randint(20, 60, 20),
        'weight_kg': np.random.randint(55, 100, 20),
        'height_cm': np.random.randint(150, 190, 20),
        'is_male': np.random.choice([True, False], 20)
    })

# Example usage:
#gt_data = create_ground_truth_data()
#evaluation = evaluate_system_accuracy(gt_data, test_dir='path/to/images')

In [ ]:
def test_analysis_on_image_with_metrics(image_path, known_hr=None, known_systolic=None, known_diastolic=None):
    """Test the heart rate analysis on a sample image with enhanced metrics"""
    # Load the image
    img = cv2.imread(image_path)
    if img is None:
        print(f"Failed to load image: {image_path}")
        return
    
    # Display the image
    plt.figure(figsize=(10, 8))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title("Original Image")
    plt.axis('off')
    plt.show()
    
    # Initialize face cascade
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    
    # Extract face regions
    face_info = extract_face_roi(img, face_cascade)
    if face_info is None:
        print("No face detected in the image!")
        return
    
    # Display the face regions
    fig, axes = plt.subplots(1, 4, figsize=(15, 5))
    regions = ['face', 'forehead', 'left_cheek', 'right_cheek']
    titles = ['Face', 'Forehead', 'Left Cheek', 'Right Cheek']
    
    for i, region in enumerate(regions):
        axes[i].imshow(cv2.cvtColor(face_info[region], cv2.COLOR_BGR2RGB))
        axes[i].set_title(titles[i])
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Analyze the face
    analysis_result = analyze_skin_color_variations(face_info)
    
    # Create diagnostic image
    diagnostic_img = create_diagnostic_image(img, face_info)
    if diagnostic_img:
        # Convert base64 to image for display
        decoded = base64.b64decode(diagnostic_img)
        plt.figure(figsize=(10, 8))
        plt.imshow(plt.imread(io.BytesIO(decoded)))
        plt.title("Diagnostic Image")
        plt.axis('off')
        plt.show()
    
    # Display the analysis results
    print(f"Estimated Heart Rate: {analysis_result['heart_rate']} BPM")
    print(f"Heart Rate Range: {analysis_result['heart_rate_range'][0]}-{analysis_result['heart_rate_range'][1]} BPM")
    print(f"Confidence: {analysis_result['confidence']}%")
    
    # Calculate error if known heart rate is provided
    metrics = {}
    if known_hr:
        hr_error = abs(analysis_result['heart_rate'] - known_hr)
        hr_error_pct = hr_error / known_hr * 100
        hr_within_range = known_hr >= analysis_result['heart_rate_range'][0] and known_hr <= analysis_result['heart_rate_range'][1]
        
        print(f"\nKnown Heart Rate: {known_hr} BPM")
        print(f"Absolute Error: {hr_error} BPM")
        print(f"Relative Error: {hr_error_pct:.2f}%")
        print(f"Within Error Range: {'Yes' if hr_within_range else 'No'}")
        
        metrics['heart_rate'] = {
            'absolute_error': hr_error,
            'relative_error': hr_error_pct,
            'within_range': hr_within_range,
            'accuracy': 100 - hr_error_pct,
            'precision': analysis_result['heart_rate_range'][1] - analysis_result['heart_rate_range'][0]
        }
    
    # Estimate blood pressure with demographics
    age = 30  # Example age
    weight_kg = 70  # Example weight
    height_cm = 170  # Example height
    is_male = True  # Example gender
    
    sys_bp, dia_bp = estimate_bp(analysis_result['heart_rate'], age, weight_kg, height_cm, is_male)
    print(f"\nEstimated Blood Pressure: {sys_bp}/{dia_bp} mmHg")
    
    # BP error metrics if known values are provided
    if known_systolic and known_diastolic:
        sys_error = abs(sys_bp - known_systolic)
        dia_error = abs(dia_bp - known_diastolic)
        sys_error_pct = sys_error / known_systolic * 100
        dia_error_pct = dia_error / known_diastolic * 100
        
        print(f"Known Blood Pressure: {known_systolic}/{known_diastolic} mmHg")
        print(f"Systolic Error: {sys_error} mmHg ({sys_error_pct:.2f}%)")
        print(f"Diastolic Error: {dia_error} mmHg ({dia_error_pct:.2f}%)")
        
        # Clinical acceptability (within ±10 mmHg for systolic, ±5 mmHg for diastolic)
        sys_acceptable = sys_error <= 10
        dia_acceptable = dia_error <= 5
        print(f"Clinically Acceptable: {'Yes' if sys_acceptable and dia_acceptable else 'No'}")
        
        metrics['blood_pressure'] = {
            'systolic_error': sys_error,
            'diastolic_error': dia_error,
            'systolic_relative_error': sys_error_pct,
            'diastolic_relative_error': dia_error_pct,
            'clinically_acceptable': sys_acceptable and dia_acceptable
        }
    
    # Display the detailed metrics
    if 'metrics' in analysis_result:
        print("\nDetailed Metrics:")
        for key, value in analysis_result['metrics'].items():
            print(f"  {key}: {value}")
    
    # Visualize the results
    plt.figure(figsize=(15, 10))
    
    # Heart rate visualization
    plt.subplot(2, 2, 1)
    plt.bar(['Heart Rate'], [analysis_result['heart_rate']], color='blue', alpha=0.7)
    plt.errorbar(['Heart Rate'], [analysis_result['heart_rate']], 
                yerr=[[analysis_result['heart_rate'] - analysis_result['heart_rate_range'][0]], 
                      [analysis_result['heart_rate_range'][1] - analysis_result['heart_rate']]], 
                fmt='o', color='red', ecolor='red', capsize=10)
    if known_hr:
        plt.axhline(y=known_hr, color='green', linestyle='--', label=f'Known HR: {known_hr}')
        plt.legend()
    plt.title('Heart Rate Estimation')
    plt.ylabel('BPM')
    plt.ylim(max(40, min(analysis_result['heart_rate_range'][0]-10, known_hr-10 if known_hr else 0)), 
             min(140, max(analysis_result['heart_rate_range'][1]+10, known_hr+10 if known_hr else 150)))
    
    # Blood pressure visualization
    plt.subplot(2, 2, 2)
    plt.bar(['Systolic', 'Diastolic'], [sys_bp, dia_bp], color=['orange', 'green'], alpha=0.7)
    if known_systolic and known_diastolic:
        plt.plot(['Systolic', 'Diastolic'], [known_systolic, known_diastolic], 'ro-', label='Known BP')
        plt.legend()
    plt.title('Blood Pressure Estimation')
    plt.ylabel('mmHg')
    
    # Error visualization if ground truth is available
    if known_hr or (known_systolic and known_diastolic):
        plt.subplot(2, 2, 3)
        
        if known_hr:
            hr_error_data = {'Measurement': ['Heart Rate'], 
                           'Absolute Error': [metrics['heart_rate']['absolute_error']], 
                           'Relative Error (%)': [metrics['heart_rate']['relative_error']]}
            hr_df = pd.DataFrame(hr_error_data)
            hr_df = hr_df.set_index('Measurement')
            hr_df.plot(kind='bar', ax=plt.gca())
            plt.title('Heart Rate Error Metrics')
            plt.ylabel('Error Value')
            
        plt.subplot(2, 2, 4)
        if known_systolic and known_diastolic:
            bp_error_data = {'Measurement': ['Systolic', 'Diastolic'], 
                          'Absolute Error': [metrics['blood_pressure']['systolic_error'], 
                                           metrics['blood_pressure']['diastolic_error']], 
                          'Relative Error (%)': [metrics['blood_pressure']['systolic_relative_error'], 
                                              metrics['blood_pressure']['diastolic_relative_error']]}
            bp_df = pd.DataFrame(bp_error_data)
            bp_df = bp_df.set_index('Measurement')
            bp_df.plot(kind='bar', ax=plt.gca())
            plt.title('Blood Pressure Error Metrics')
            plt.ylabel('Error Value')
    
    plt.tight_layout()
    plt.show()
    
    return {
        'analysis_result': analysis_result,
        'metrics': metrics
    }

# Example usage:
# test_analysis_on_image_with_metrics('path_to_image.jpg', known_hr=72, known_systolic=120, known_diastolic=80)

In [3]:
def analyze_system_errors(results_df):
    """
    Perform detailed error analysis on the system's predictions
    
    Parameters:
    -----------
    results_df : DataFrame
        DataFrame containing test results with ground truth and estimated values
        
    Returns:
    --------
    dict
        Dictionary with error analysis results
    """
    error_analysis = {}
    
    # HR error analysis by confidence level
    if 'confidence' in results_df.columns and 'hr_error' in results_df.columns:
        # Create confidence bins
        results_df['confidence_bin'] = pd.cut(results_df['confidence'], 
                                             bins=[0, 60, 70, 80, 90, 100], 
                                             labels=['<60%', '60-70%', '70-80%', '80-90%', '90-100%'])
        
        # Group by confidence bin and calculate mean error
        hr_error_by_confidence = results_df.groupby('confidence_bin')['hr_error'].agg(['mean', 'median', 'count'])
        
        error_analysis['hr_error_by_confidence'] = hr_error_by_confidence
        
        # Visualize
        plt.figure(figsize=(12, 6))
        sns.barplot(x=results_df['confidence_bin'], y=results_df['hr_error'])
        plt.title('Heart Rate Error by Confidence Level')
        plt.xlabel('Confidence')
        plt.ylabel('Mean Absolute Error (BPM)')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
        
        # Print the results
        print("Heart Rate Error by Confidence Level:")
        print(hr_error_by_confidence)
        print()
    
    # HR error distribution
    if 'hr_error' in results_df.columns:
        plt.figure(figsize=(12, 6))
        
        plt.subplot(1, 2, 1)
        sns.histplot(results_df['hr_error'], kde=True, bins=10)
        plt.axvline(results_df['hr_error'].mean(), color='red', linestyle='--', 
                   label=f'Mean: {results_df["hr_error"].mean():.2f}')
        plt.axvline(results_df['hr_error'].median(), color='green', linestyle='--', 
                   label=f'Median: {results_df["hr_error"].median():.2f}')
        plt.title('Heart Rate Error Distribution')
        plt.xlabel('Absolute Error (BPM)')
        plt.ylabel('Frequency')
        plt.legend()
        
        # Calculate error percentiles
        hr_error_percentiles = np.percentile(results_df['hr_error'], [25, 50, 75, 90, 95, 99])
        error_analysis['hr_error_percentiles'] = {
            '25th': hr_error_percentiles[0],
            '50th': hr_error_percentiles[1],
            '75th': hr_error_percentiles[2],
            '90th': hr_error_percentiles[3],
            '95th': hr_error_percentiles[4],
            '99th': hr_error_percentiles[5]
        }
        
        # Print percentiles
        print("Heart Rate Error Percentiles:")
        for pct, value in error_analysis['hr_error_percentiles'].items():
            print(f"  {pct} percentile: {value:.2f} BPM")
        
        # Cumulative error distribution
        plt.subplot(1, 2, 2)
        sns.ecdfplot(results_df['hr_error'])
        plt.axvline(x=5, color='red', linestyle='--', 
                   label=f'Clinical Threshold (±5 BPM): {(results_df["hr_error"] <= 5).mean()*100:.1f}%')
        plt.title('Cumulative Heart Rate Error Distribution')
        plt.xlabel('Absolute Error (BPM)')
        plt.ylabel('Cumulative Proportion')
        plt.legend()
        
        plt.tight_layout()
        plt.show()
    
    # Blood pressure error analysis (similar approach)
    if 'systolic_error' in results_df.columns and 'diastolic_error' in results_df.columns:
        plt.figure(figsize=(12, 6))
        
        plt.subplot(1, 2, 1)
        sns.histplot(results_df['systolic_error'], kde=True, color='red', alpha=0.5, label='Systolic')
        sns.histplot(results_df['diastolic_error'], kde=True, color='blue', alpha=0.5, label='Diastolic')
        plt.title('Blood Pressure Error Distribution')
        plt.xlabel('Absolute Error (mmHg)')
        plt.ylabel('Frequency')
        plt.legend()
        
        plt.subplot(1, 2, 2)
        sns.ecdfplot(results_df['systolic_error'], label='Systolic')
        sns.ecdfplot(results_df['diastolic_error'], label='Diastolic')
        plt.axvline(x=10, color='red', linestyle='--', 
                   label=f'Systolic Threshold (±10 mmHg): {(results_df["systolic_error"] <= 10).mean()*100:.1f}%')
        plt.axvline(x=5, color='blue', linestyle='--', 
                   label=f'Diastolic Threshold (±5 mmHg): {(results_df["diastolic_error"] <= 5).mean()*100:.1f}%')
        plt.title('Cumulative Blood Pressure Error Distribution')
        plt.xlabel('Absolute Error (mmHg)')
        plt.ylabel('Cumulative Proportion')
        plt.legend()
        
        plt.tight_layout()
        plt.show()
        
        # Calculate BP error percentiles
        sys_error_percentiles = np.percentile(results_df['systolic_error'], [50, 75, 90])
        dia_error_percentiles = np.percentile(results_df['diastolic_error'], [50, 75, 90])
        
        error_analysis['bp_error_percentiles'] = {
            'systolic': {
                '50th': sys_error_percentiles[0],
                '75th': sys_error_percentiles[1],
                '90th': sys_error_percentiles[2]
            },
            'diastolic': {
                '50th': dia_error_percentiles[0],
                '75th': dia_error_percentiles[1],
                '90th': dia_error_percentiles[2]
            }
        }
        
        print("\nBlood Pressure Error Percentiles:")
        print("  Systolic:")
        for pct, value in error_analysis['bp_error_percentiles']['systolic'].items():
            print(f"    {pct} percentile: {value:.2f} mmHg")
        print("  Diastolic:")
        for pct, value in error_analysis['bp_error_percentiles']['diastolic'].items():
            print(f"    {pct} percentile: {value:.2f} mmHg")
    
    return error_analysis

# Example usage:
# error_analysis = analyze_system_errors(results_df)